### 01_raw_utils : Task 1 Library

Raw-layer utilities: `create_schemas`, `clean_cols`, `load_raw_cust_data`, `load_raw_products_data`, `load_raw_orders_data`. Also documents the loading assumptions. Imported by `01_raw_data_ingestion` via `%run`; its code cells are executed directly by the test suite (`tests/conftest.py`).

In [0]:
import pyspark.sql.functions as F

In [0]:
def create_schemas():
    """Create the raw, enriched and curated schemas if they do not already exist."""
    spark.sql("CREATE SCHEMA IF NOT EXISTS raw")        # landing zone: source data as loaded
    spark.sql("CREATE SCHEMA IF NOT EXISTS enriched")   # cleaned and typed data
    spark.sql("CREATE SCHEMA IF NOT EXISTS curated")    # business-ready, joined data

In [0]:
def clean_cols(df):
    """
    Normalize all column names of a DataFrame: strip surrounding whitespace,
    lowercase, and replace spaces and hyphens with underscores.

    Returns a new DataFrame with the renamed columns.
    """
    for c in df.columns:
        new_c = c.strip().lower().replace(" ", "_").replace("-", "_")
        df = df.withColumnRenamed(c, new_c)
    return df

#### Task 1 : Raw Data Loads

Run in `01_raw_data_ingestion`.

Raw Customer Data - `load_raw_cust_data`

In [0]:
def load_raw_cust_data():
    """
    Load the raw customer data from Customer.xlsx into raw.customers.

    Reads the "Worksheet" sheet using its first row as the header, normalizes
    the column names, and overwrites the raw.customers table (schema included).
    """
    raw_cust_df = (
        spark.read
          .option("headerRows", 1)             # first sheet row holds the column names
          .option("dataAddress", "Worksheet")   # sheet to read within the workbook
        .excel("/Volumes/workspace/default/peiassignmentdata/Customer.xlsx")
    )
    raw_cust_df = clean_cols(raw_cust_df)   # normalize column names before persisting
    raw_cust_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("raw.customers")

Raw Products Data - `load_raw_product_data`

In [0]:
def load_raw_products_data():
    """
    Load the raw product data from Products.csv into raw.products.

    Reads the CSV with a header row, honoring embedded quotes and escapes, and
    keeps every column as a string (no schema inference) so the raw layer
    preserves source values verbatim. Overwrites raw.products (schema included).
    """
    raw_product_df = (
        spark.read.option("header", "true")
        .option("escape", "\"")        # handle escaped quotes inside values
        .option("quote", "\"")         # double-quote is the quote character
        .option("inferSchema", "false")   # keep all columns as string in the raw layer
        .csv("/Volumes/workspace/default/peiassignmentdata/Products.csv")
    )
    raw_product_df = clean_cols(raw_product_df)   # normalize column names before persisting
    raw_product_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("raw.products")

Raw Orders Data - `load_raw_orders_data`

In [0]:
def load_raw_orders_data():
    """
    Load the raw orders data from Orders.json into raw.orders.

    Reads the JSON in multiline mode because each order record spans multiple
    lines. Column names are normalized and the raw.orders table is overwritten
    (schema included).
    """
    orders_raw = (
        spark.read
        .option("multiline", "true")   # JSON records span multiple lines
        .json("/Volumes/workspace/default/peiassignmentdata/Orders.json")
    )
    orders_raw = clean_cols(orders_raw)   # normalize column names before persisting
    orders_raw.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("raw.orders")